# 01 · Define & Explore — humanization strategies, the trade-off, + the hello-world

**Standard slot:** *define & explore.* **For Project 16 this means:** understand the **non-human
therapeutic antibody** you will humanize and the **human germline frameworks** you will graft onto,
learn the three humanization strategies (**CDR grafting**, **resurfacing**, **germline-content
optimization**), fix the metrics table, and run the **mock** humanization hello-world end-to-end (D0).

Run `00_setup.ipynb` first in this session. Everything here runs with **no GPU** on the deterministic
`mock` backend. Compute for this whole project is genuinely **light** (humanness scoring + IgFold +
ΔΔG proxies are free-tier Colab **T4** friendly) — switch each `tool="mock"` to the real backend on
Colab when you are ready.

## The problem in one screen

A **non-human antibody** (murine, or a murine/human **chimera**) used as a therapeutic triggers an
**anti-drug-antibody (ADA)** response: the patient's immune system recognizes the non-human sequence as
foreign, clears the drug, and can cause adverse reactions. **Humanization** rewrites the antibody so it
looks like a **human germline antibody** (low ADA risk) while keeping the original **CDRs** that confer
binding. It is a **regulatory necessity** for non-human therapeutic candidates.

The three classic strategies:
- **CDR grafting** — transplant the non-human **CDR loops** onto a **human germline framework**
  (FR1..FR4). Maximally human framework, but the new framework residues that *support* the CDR loops
  (the **Vernier zone**) often perturb the loops → **affinity / stability loss**. The fix is selective
  **back-mutation** of Vernier-zone residues (restore the parental residue).
- **Resurfacing (veneering)** — keep the non-human framework **core**, mutate only the **surface-exposed**
  framework residues to human identity. Changes far fewer residues (lower ΔΔG risk) but achieves **less
  humanness**.
- **Germline-content optimization** — push the sequence toward the nearest human **germline** content,
  scored by humanness tools (OASis / Hu-mAb / T20 / AbLang).

**The central tension (this whole project):** *humanness ↔ stability is a trade-off.* More framework
humanization usually means more mutations means higher destabilization (**ΔΔG**). The deliverable is
**humanized variants + the trade-off analysis + the back-mutations needed + a validation plan** — NOT
"a humanized antibody". A computational design is a **hypothesis**; expression is not function; a
humanness score is not a guaranteed low-ADA outcome.

## The metrics table (what we will measure and filter on)

| Metric | Range | Means | Does **not** mean | Direction |
|--------|-------|-------|-------------------|-----------|
| humanness (OASis-like) | 0–1 | fraction of 9-mers matching human repertoire (proxy) | guaranteed low immunogenicity | higher = more human |
| humanness (T20-like) | 0–100 | rescaled human-likeness (proxy) | a clinical ADA prediction | higher = more human |
| germline_id | label | nearest human germline (proxy) | a validated germline call | human-like wanted |
| ΔΔG (proxy) | a.u. (signed) | predicted destabilization of the graft vs parental | a real FoldX/Rosetta kcal/mol | **lower** = more stable (>0 destabilizing) |
| n_framework_mutations | count | how many FR residues changed from parental | humanness by itself | trade-off knob |
| pLDDT (IgFold/AF2) | 0–100 | model local confidence of the Fv | stability / affinity | higher = more confident |
| scRMSD | Å | designed-vs-predicted backbone (self-consistency) | binding | ≤ 3.0 (antibody) |

The shared filter uses `filtering_pipeline.DEFAULT_CUTOFFS["antibody"]` (scRMSD ≤ 3.0, pLDDT ≥ 70,
pae_interaction ≤ 12); we map humanness and the ΔΔG proxy onto it in notebook 03. **Humanness and ΔΔG
here are TEACHING HEURISTICS, not the validated tools (OASis/Hu-mAb/T20/AbLang; FoldX/Rosetta)** — see
`MANUAL.md §2` and `humanization_tools.py`. Never report a heuristic number as a real result.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Pick your non-human antibody + human germline framework

These are the two fixed inputs to the campaign. **The student supplies a real published murine /
chimeric therapeutic antibody** (VH/VL sequence) and a **human germline framework** from IMGT/OAS — the
placeholders below are TEACHING sequences, **not** a specific real antibody. **Verify your antibody
sequence and your germline choice in Week 1** (`data/README.md`).

In [ ]:
from humanization_tools import NONHUMAN_AB, HUMAN_FRAMEWORK, split_regions, parental_sequence

# --- The non-human (e.g., murine/chimeric) antibody you are humanizing (TEACHING placeholder) ---
# VERIFY/replace with your real published therapeutic antibody VH (and repeat for VL) in Week 1.
print("NON-HUMAN ANTIBODY (to humanize):", NONHUMAN_AB["name"], "chain", NONHUMAN_AB["chain"])
for k in ("FR1", "CDR1", "FR2", "CDR2", "FR3", "CDR3", "FR4"):
    print(f"  {k:5s}: {NONHUMAN_AB[k]}")

# --- The human germline framework you graft the CDRs ONTO (TEACHING placeholder) ---
# VERIFY/replace with the exact IMGT human germline you select (e.g., an IGHV3 family member).
print("\nHUMAN GERMLINE FRAMEWORK (graft target):", HUMAN_FRAMEWORK["name"])
for k in ("FR1", "FR2", "FR3", "FR4"):
    print(f"  {k:5s}: {HUMAN_FRAMEWORK[k]}")

parent = parental_sequence(NONHUMAN_AB)
print("\nparental VH sequence length:", len(parent), "aa  (this is the ΔΔG / humanness BASELINE)")

## The Vernier zone (why grafting costs affinity)

The **Vernier zone** is the set of framework residues that pack against and *position* the CDR loops
(Foote & Winter 1992). When CDR grafting replaces a Vernier residue with the human one, the CDR loop can
shift → lost affinity/stability. Those positions are the prime **back-mutation** candidates: restore the
parental (non-human) residue to rescue binding, paying a small humanness cost. `VERNIER_ZONE` below uses
per-region positions approximating the canonical set (teaching-grade — map true Kabat/IMGT Vernier
positions with ANARCI for a real run).

In [ ]:
from humanization_tools import VERNIER_ZONE
print("Vernier-zone positions (per framework region, 0-based — TEACHING approximation):")
for region, positions in VERNIER_ZONE.items():
    print(f"  {region}: {positions}")
print("\nThese are the framework residues that support the CDR loops — the prime back-mutation targets.")

## Humanization hello-world (mock backend, no GPU)

Graft the non-human CDRs onto the human framework, score humanness + the ΔΔG proxy, and list the Vernier
back-mutations the graft suggests. This proves the plumbing
(graft → humanness → ΔΔG → back-mutations) before any real run. **Every number below is SYNTHETIC —
never report mock numbers as real.**

In [ ]:
from humanization_tools import (graft_cdrs, vernier_backmutations, score_variants,
                                  humanness_score, ddg_predict)

# 1) Graft the non-human CDRs onto the human germline framework (no back-mutations yet).
graft = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, scheme="kabat", tool="mock")

# 2) Score humanness + the ΔΔG proxy (vs the parental sequence).
score_variants([graft], parent, tool="mock")

# 3) The Vernier back-mutations this graft suggests (restore parental residues at CDR-support sites).
bms = vernier_backmutations(graft, NONHUMAN_AB, HUMAN_FRAMEWORK)

print("variant_id        :", graft.variant_id, "(method:", graft.method + ")")
print("VH length         :", len(graft.sequence), "aa")
print("framework muts     :", graft.n_framework_mutations, "(human FR residues differing from parental)")
print("humanness (heur)  : OASis-like", graft.oasis_like, "| T20-like", graft.t20_like,
      "| germline", graft.germline_id)
print("ΔΔG proxy (heur)  :", graft.ddg_kcal_mol, "(>0 = destabilizing; NOT real kcal/mol)")
print("synthetic         :", graft.synthetic, "->", graft.notes)
print("\nVernier back-mutation suggestions:")
for b in bms:
    print("  ", b["token"], "-", b["rationale"])

## A first look at the trade-off

Even at hello-world scale you can see the tension. Compare the **bare graft** (maximally human, costly)
to a graft **with Vernier back-mutations** (a little less human, more stable). Notebook 04 turns this
into the full trade-off figure across many variants and against the resurfacing alternative. **Mock
numbers are SYNTHETIC** — the *shape* of the trade-off is the teaching point, not the values.

In [ ]:
# Re-graft WITH the Vernier back-mutations applied — the affinity/stability rescue.
tokens = tuple(b["token"] for b in bms)
graft_bm = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, back_mutations=tokens, tool="mock")
score_variants([graft_bm], parent, tool="mock")

print(f"{'variant':28s} {'humanness':>10s} {'ΔΔG proxy':>10s} {'back-muts':>10s}")
print(f"{graft.variant_id:28s} {graft.oasis_like:>10} {graft.ddg_kcal_mol:>10} {len(graft.back_mutations):>10}")
print(f"{graft_bm.variant_id:28s} {graft_bm.oasis_like:>10} {graft_bm.ddg_kcal_mol:>10} {len(graft_bm.back_mutations):>10}")
print("\nExpected DIRECTION (SYNTHETIC values): back-mutations trade a little humanness for lower ΔΔG.")
print("This humanness<->stability trade-off IS the deliverable — quantify it in nb 04.")

## D0 checklist
- [ ] 1-page **problem statement**: the **non-human antibody** (verified, published), the **human
      germline framework(s)** you will graft onto, the humanization **strategy** (grafting vs
      resurfacing vs germline-content), and **measurable** success criteria (target humanness band +
      acceptable ΔΔG / retained-binding bar).
- [ ] Verified your antibody sequence + germline framework choice (IMGT/OAS); placeholders replaced.
- [ ] Metric table understood, including the "does not mean" column and that humanness + ΔΔG here are
      **heuristics**, not validated tools.
- [ ] Mock humanization hello-world run; graft + Vernier back-mutations + SYNTHETIC humanness/ΔΔG printed.
- [ ] `LOG.md` entry (seed, what you ran).

**Next:** `02_generate.ipynb` — graft CDRs onto candidate frameworks → variants CSV (mock now; real
AbLang/IgFold on Colab T4).